# Preparacion de salida para carga al mosaico

Notebook minimo para validar imagenes nuevas antes de copiarlas a la estructura del datastore. La entrada es una carpeta de imagenes y el feature class de sectores. La salida es un manifiesto con el nombre esperado por cruce geografico.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

# PARAMETROS MINIMOS
PATH_INPUT_IMAGENES = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260519_Geosupport"
PATH_FC_INDICE_VUELOS_IMGS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
SECTOR_FIELD_INDICE_VUELOS = "Sector"
QUERY_INDICE_VUELOS =  "Sensor <> 'DJI MATRICE 350 RTK'"  # Ejemplo: "Estado = 'Activo'"

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modulo auditoria:", mosaic_audit.__file__)
print("Version logica:", AUDIT_LOGIC_VERSION)
print("Input imagenes:", PATH_INPUT_IMAGENES)
print("Feature sectores:", PATH_FC_INDICE_VUELOS_IMGS)
print("Campo sector:", SECTOR_FIELD_INDICE_VUELOS)
print("Query sectores:", QUERY_INDICE_VUELOS)
print("Salida:", OUTPUT_DIR)

Modulo auditoria: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\core\mosaic_image_audit.py
Version logica: 2026-06-15-spatial-sector-preserve-token
Input imagenes: \\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260519_Geosupport
Feature sectores: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
Campo sector: Sector
Query sectores: Sensor <> 'DJI MATRICE 350 RTK'
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_142946


## 1. Buscar imagenes

La busqueda es recursiva. Solo se preparan `tif` y `tiff` para la carga al mosaico.

In [2]:
input_images_df = scan_input_images(PATH_INPUT_IMAGENES)
ortho_images_df = input_images_df[input_images_df["extension"].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()

print(f"Archivos encontrados: {len(input_images_df)}")
print(f"Imagenes TIF/TIFF para evaluar: {len(ortho_images_df)}")

display(input_images_df.groupby("extension").size().reset_index(name="count"))
display(ortho_images_df[["file_name", "relative_path", "size_mb", "modified_at"]].head(20))

Archivos encontrados: 1392
Imagenes TIF/TIFF para evaluar: 33


,extension,count
0,.jpg,1359
1,.tif,33


,file_name,relative_path,size_mb,modified_at
1359,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,SOLO_TIF_19May26\GEOSP-TRN-002511_GS_Ortofoto ...,79.159,2026-06-05 11:44:49
1360,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,SOLO_TIF_19May26\GEOSP-TRN-002545_GS_ORTOFOTO_...,11.012,2026-06-05 11:44:50
1361,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,SOLO_TIF_19May26\GEOSP-TRN-002546_GS_ORTOFOTO_...,762.001,2026-06-05 11:44:57
1362,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,SOLO_TIF_19May26\GEOSP-TRN-002555_GS_Ortofoto_...,640.965,2026-06-05 11:45:11
1363,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,SOLO_TIF_19May26\GEOSP-TRN-002591_GS_Ortofoto ...,63.628,2026-06-05 11:45:18
1364,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,SOLO_TIF_19May26\GEOSP-TRN-002592_GS_Ortofoto_...,102.047,2026-06-05 11:45:22
1365,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,SOLO_TIF_19May26\GEOSP-TRN-002593_GS_ORTOFOTO_...,156.844,2026-06-05 11:45:24
1366,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,SOLO_TIF_19May26\GEOSP-TRN-002603_ORTOFOTO_COR...,71.190,2026-06-05 11:45:29
1367,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,SOLO_TIF_19May26\GEOSP-TRN-002604_GS_Ortofoto_...,3217.563,2026-06-05 11:45:59
1368,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,SOLO_TIF_19May26\GEOSP-TRN-002606_GS_Ortofoto_...,264.282,2026-06-05 11:46:07


## 2. Calcular sector geografico y nombre esperado

El sector viene solo del cruce espacial. Si una imagen cruza mas de un sector, se usa el sector con mayor porcentaje de interseccion. El texto del sector se conserva desde el feature class y solo se reemplazan espacios por `_`.

In [3]:
spatial_matches_df = calculate_spatial_sector_matches(
    ortho_images_df,
    PATH_FC_INDICE_VUELOS_IMGS,
    sector_field=SECTOR_FIELD_INDICE_VUELOS,
    where_clause=QUERY_INDICE_VUELOS,
)

prepared_df = add_expected_names_with_spatial_sector(
    ortho_images_df,
    spatial_matches_df,
)

display(spatial_matches_df["spatial_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "spatial_status"}))
display(prepared_df["rename_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "rename_status"}))

,spatial_status,count
0,ok,33


,rename_status,count
0,ok,32
1,sin_fecha,1


## 3. Preparar manifiesto de salida

`ready_for_datastore` indica registros listos para validar y luego copiar. Este notebook no copia archivos.

In [4]:
manifest_df = prepared_df.copy()

manifest_df["destination_date_folder"] = manifest_df["expected_date_token"].map(
    lambda value: "_".join(str(value).split("_")[:2]) if pd.notna(value) and value else None
)
manifest_df["ready_for_datastore"] = manifest_df["rename_status"].eq("ok")
manifest_df["duplicate_expected_file_name"] = (
    manifest_df["expected_file_name"].notna()
    & manifest_df.duplicated("expected_file_name", keep=False)
)

def build_review_reason(row):
    reasons = []
    if row.get("rename_status") != "ok":
        reasons.append(str(row.get("rename_status")))
    if row.get("duplicate_expected_file_name"):
        reasons.append("nombre_esperado_duplicado")
    if row.get("spatial_overlap_count", 0) and row.get("spatial_overlap_count", 0) > 1:
        reasons.append("cruza_multiples_sectores")
    return "|".join(reasons) if reasons else None

manifest_df["review_reason"] = manifest_df.apply(build_review_reason, axis=1)

output_columns = [
    "ready_for_datastore",
    "review_reason",
    "path",
    "relative_path",
    "file_name",
    "expected_file_name",
    "expected_name",
    "expected_date_token",
    "destination_date_folder",
    "expected_sector",
    "sector_source",
    "rename_status",
    "spatial_status",
    "spatial_sector_raw",
    "spatial_sector",
    "spatial_overlap_pct",
    "spatial_overlap_count",
    "spatial_all_matches",
    "duplicate_expected_file_name",
    "size_mb",
    "modified_at",
]
output_columns = [column for column in output_columns if column in manifest_df.columns]
manifest_output_df = manifest_df[output_columns].copy()

print(f"Listas para validar/copiar: {int(manifest_output_df['ready_for_datastore'].sum())}")
print(f"Requieren revision: {int((~manifest_output_df['ready_for_datastore']).sum())}")
print(f"Nombres esperados duplicados: {int(manifest_output_df['duplicate_expected_file_name'].sum())}")

display(manifest_output_df.head(30))
display(manifest_output_df[manifest_output_df["review_reason"].notna()].head(30))

Listas para validar/copiar: 32
Requieren revision: 1
Nombres esperados duplicados: 10


,ready_for_datastore,review_reason,path,relative_path,file_name,expected_file_name,expected_name,expected_date_token,destination_date_folder,expected_sector,...,rename_status,spatial_status,spatial_sector_raw,spatial_sector,spatial_overlap_pct,spatial_overlap_count,spatial_all_matches,duplicate_expected_file_name,size_mb,modified_at
0,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002511_GS_Ortofoto ...,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,26_05_01,26_05,Estacion_Cabecera,...,ok,ok,Estacion_Cabecera,Estacion_Cabecera,46.146846,5,Estacion_Cabecera:46.15|Estacion_Cabecera:46.0...,False,79.159,2026-06-05 11:44:49
1,True,nombre_esperado_duplicado|cruza_multiples_sect...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002545_GS_ORTOFOTO_...,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro,26_05_06,26_05,Subestacion-El-Mauro,...,ok,ok,Subestacion-El-Mauro,Subestacion-El-Mauro,99.999972,31,Subestacion-El-Mauro:100.00|Subestacion-El-Mau...,True,11.012,2026-06-05 11:44:50
2,True,nombre_esperado_duplicado|cruza_multiples_sect...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002546_GS_ORTOFOTO_...,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro,26_05_06,26_05,Subestacion-El-Mauro,...,ok,ok,Subestacion-El-Mauro,Subestacion-El-Mauro,54.915439,37,Subestacion-El-Mauro:54.92|Subestacion-El-Maur...,True,762.001,2026-06-05 11:44:57
3,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002555_GS_Ortofoto_...,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,26_05_06,26_05,TORRE_E85_A_E_125,...,ok,ok,TORRE_E85_A_E_125,TORRE_E85_A_E_125,49.100137,37,TORRE_E85_A_E_125:49.10|TORRE_E85_A_E_125:48.9...,False,640.965,2026-06-05 11:45:11
4,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002591_GS_Ortofoto ...,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,26_05_10,26_05,ED1,...,ok,ok,ED1,ED1,44.184240,10,ED1:44.18|ED1:43.65|ED1-IIFF7-DME8:43.40|ED1-I...,False,63.628,2026-06-05 11:45:18
5,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002592_GS_Ortofoto_...,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto.tif,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,26_05_10,26_05,Helipuerto,...,ok,ok,Helipuerto,Helipuerto,54.779015,5,Helipuerto:54.78|Helipuerto:54.78|Helipuerto:5...,False,102.047,2026-06-05 11:45:22
6,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002593_GS_ORTOFOTO_...,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armad...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,26_05_10,26_05,Patio-19B-y-Armado,...,ok,ok,Patio-19B-y-Armado,Patio-19B-y-Armado,44.894080,15,Patio-19B-y-Armado:44.89|Patio-19B-y-Armado:43...,False,156.844,2026-06-05 11:45:24
7,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002603_ORTOFOTO_COR...,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,26_05_10,26_05,DME9-PA12-IIFF8,...,ok,ok,DME9-PA12-IIFF8,DME9-PA12-IIFF8,33.472362,11,DME9-PA12-IIFF8:33.47|EM2:32.56|EM2:31.98|DME9...,False,71.190,2026-06-05 11:45:29
8,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_

,ready_for_datastore,review_reason,path,relative_path,file_name,expected_file_name,expected_name,expected_date_token,destination_date_folder,expected_sector,...,rename_status,spatial_status,spatial_sector_raw,spatial_sector,spatial_overlap_pct,spatial_overlap_count,spatial_all_matches,duplicate_expected_file_name,size_mb,modified_at
0,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002511_GS_Ortofoto ...,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,26_05_01,26_05,Estacion_Cabecera,...,ok,ok,Estacion_Cabecera,Estacion_Cabecera,46.146846,5,Estacion_Cabecera:46.15|Estacion_Cabecera:46.0...,False,79.159,2026-06-05 11:44:49
1,True,nombre_esperado_duplicado|cruza_multiples_sect...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002545_GS_ORTOFOTO_...,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro,26_05_06,26_05,Subestacion-El-Mauro,...,ok,ok,Subestacion-El-Mauro,Subestacion-El-Mauro,99.999972,31,Subestacion-El-Mauro:100.00|Subestacion-El-Mau...,True,11.012,2026-06-05 11:44:50
2,True,nombre_esperado_duplicado|cruza_multiples_sect...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002546_GS_ORTOFOTO_...,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro,26_05_06,26_05,Subestacion-El-Mauro,...,ok,ok,Subestacion-El-Mauro,Subestacion-El-Mauro,54.915439,37,Subestacion-El-Mauro:54.92|Subestacion-El-Maur...,True,762.001,2026-06-05 11:44:57
3,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002555_GS_Ortofoto_...,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,26_05_06,26_05,TORRE_E85_A_E_125,...,ok,ok,TORRE_E85_A_E_125,TORRE_E85_A_E_125,49.100137,37,TORRE_E85_A_E_125:49.10|TORRE_E85_A_E_125:48.9...,False,640.965,2026-06-05 11:45:11
4,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002591_GS_Ortofoto ...,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,26_05_10,26_05,ED1,...,ok,ok,ED1,ED1,44.184240,10,ED1:44.18|ED1:43.65|ED1-IIFF7-DME8:43.40|ED1-I...,False,63.628,2026-06-05 11:45:18
5,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002592_GS_Ortofoto_...,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto.tif,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,26_05_10,26_05,Helipuerto,...,ok,ok,Helipuerto,Helipuerto,54.779015,5,Helipuerto:54.78|Helipuerto:54.78|Helipuerto:5...,False,102.047,2026-06-05 11:45:22
6,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002593_GS_ORTOFOTO_...,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armad...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,26_05_10,26_05,Patio-19B-y-Armado,...,ok,ok,Patio-19B-y-Armado,Patio-19B-y-Armado,44.894080,15,Patio-19B-y-Armado:44.89|Patio-19B-y-Armado:43...,False,156.844,2026-06-05 11:45:24
7,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_19May26\GEOSP-TRN-002603_ORTOFOTO_COR...,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,26_05_10,26_05,DME9-PA12-IIFF8,...,ok,ok,DME9-PA12-IIFF8,DME9-PA12-IIFF8,33.472362,11,DME9-PA12-IIFF8:33.47|EM2:32.56|EM2:31.98|DME9...,False,71.190,2026-06-05 11:45:29
8,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_

## 4. Exportar outputs

Los archivos exportados son la base para validar antes de copiar al datastore y cargar al mosaico.

In [5]:
summary_df = pd.DataFrame(
    [
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "audit_logic_version", "value": AUDIT_LOGIC_VERSION},
        {"metric": "input_folder", "value": PATH_INPUT_IMAGENES},
        {"metric": "sector_feature_class", "value": PATH_FC_INDICE_VUELOS_IMGS},
        {"metric": "sector_field", "value": SECTOR_FIELD_INDICE_VUELOS},
        {"metric": "sector_query", "value": QUERY_INDICE_VUELOS},
        {"metric": "input_files_count", "value": len(input_images_df)},
        {"metric": "ortho_images_count", "value": len(ortho_images_df)},
        {"metric": "ready_for_datastore_count", "value": int(manifest_output_df["ready_for_datastore"].sum())},
        {"metric": "review_required_count", "value": int((~manifest_output_df["ready_for_datastore"]).sum())},
        {"metric": "duplicate_expected_file_name_count", "value": int(manifest_output_df["duplicate_expected_file_name"].sum())},
    ]
)

for status, count in spatial_matches_df["spatial_status"].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {"metric": f"spatial_status_{status}", "value": int(count)}

for status, count in manifest_df["rename_status"].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {"metric": f"rename_status_{status}", "value": int(count)}

summary_csv = OUTPUT_DIR / "00_summary.csv"
input_csv = OUTPUT_DIR / "01_input_images.csv"
spatial_csv = OUTPUT_DIR / "02_spatial_matches.csv"
manifest_csv = OUTPUT_DIR / "03_manifest_carga_mosaico.csv"
ready_csv = OUTPUT_DIR / "04_ready_for_datastore.csv"
review_csv = OUTPUT_DIR / "05_review_required.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
input_images_df.to_csv(input_csv, index=False, encoding="utf-8-sig")
spatial_matches_df.to_csv(spatial_csv, index=False, encoding="utf-8-sig")
manifest_output_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
manifest_output_df[manifest_output_df["ready_for_datastore"]].to_csv(ready_csv, index=False, encoding="utf-8-sig")
manifest_output_df[~manifest_output_df["ready_for_datastore"]].to_csv(review_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Outputs exportados en:", OUTPUT_DIR)
print("Manifest principal:", manifest_csv)

,metric,value
0,run_timestamp,20260615_142946
1,audit_logic_version,2026-06-15-spatial-sector-preserve-token
2,input_folder,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...
3,sector_feature_class,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...
4,sector_field,Sector
5,sector_query,Sensor <> 'DJI MATRICE 350 RTK'
6,input_files_count,1392
7,ortho_images_count,33
8,ready_for_datastore_count,32
9,review_required_count,1


Outputs exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_142946
Manifest principal: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_142946\03_manifest_carga_mosaico.csv
